In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# ==========================================
# 1. KAGGLE AUTHENTICATION & DOWNLOAD
# ==========================================
# IMPORTANT: Put your Kaggle credentials here
kaggle_creds = {
    "username": "UNAME",
    "key": "KEY"
}

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)
os.chmod('/root/.kaggle/kaggle.json', 600)

print("Downloading the Balanced Malware Dataset...")
!pip install -q kaggle
!kaggle datasets download -d satyaprakash138/balanced-malware-image-dataset
!unzip -q balanced-malware-image-dataset.zip -d /content/binary_dataset
print("Dataset ready!\n")

# ==========================================
# 2. CONFIGURATION & DATA LOADING
# ==========================================
# This Kaggle dataset already has organized 'train' and 'val' folders!
TRAIN_DIR = '/content/binary_dataset/Data/train'
VAL_DIR = '/content/binary_dataset/Data/val'
MODEL_SAVE_PATH = '/content/drive/MyDrive/Malware_Project/binary_malware_scanner.pth'

BATCH_SIZE = 32
TOTAL_EPOCHS = 10
UNFREEZE_EPOCH = 5

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=transform)
val_dataset = datasets.ImageFolder(root=VAL_DIR, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# PyTorch assigns 0 and 1 alphabetically.
# 'MaliciousImages' = 0 | 'NormalImages' = 1
print(f"Class Mapping: {train_dataset.class_to_idx}")

# ==========================================
# 3. MODEL SETUP (BINARY)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()

for param in model.features.parameters():
    param.requires_grad = False
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)

# ==========================================
# 4. TWO-PHASE TRAINING LOOP
# ==========================================
for epoch in range(1, TOTAL_EPOCHS + 1):
    if epoch == UNFREEZE_EPOCH:
        print("\n*** EPOCH 5: Unfreezing Backbone & Applying Differential LRs ***\n")
        for param in model.features.parameters():
            param.requires_grad = True
        optimizer = optim.Adam([
            {'params': model.features.parameters(), 'lr': 1e-4},
            {'params': model.classifier.parameters(), 'lr': 1e-3}
        ])

    # --- Training Phase ---
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    loop = tqdm(train_loader, leave=False, desc=f"Epoch {epoch} Train")
    for inputs, labels in loop:
        inputs = inputs.to(device)
        labels = labels.unsqueeze(1).float().to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        predicted = (outputs > 0.0).float()
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100. * correct / total

    # --- Validation Phase (The Truth Check) ---
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.unsqueeze(1).float().to(device)
            outputs = model(inputs)

            predicted = (outputs > 0.0).float()
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = 100. * val_correct / val_total
    print(f"Epoch {epoch} -> Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

# ==========================================
# 5. SAVE TO GOOGLE DRIVE
# ==========================================
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)
torch.save(model.state_dict(), MODEL_SAVE_PATH)
print(f"\nModel saved perfectly to: {MODEL_SAVE_PATH}")

Dataset URL: https://www.kaggle.com/datasets/satyaprakash138/balanced-malware-image-dataset
License(s): apache-2.0
100% 1.13G/1.13G [00:20<00:00, 59.6MB/s]

Dataset ready!

Class Mapping: {'MaliciousImages': 0, 'NormalImages': 1}
Training on: cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 128MB/s] 


Epoch 1 Train:   0%|          | 0/534 [00:00<?, ?it/s]

Epoch 1 -> Train Acc: 89.36% | Val Acc: 93.85%


Epoch 2 Train:   0%|          | 0/534 [00:00<?, ?it/s]

...

Epoch 8 -> Train Acc: 99.33% | Val Acc: 98.97%


Epoch 9 Train:   0%|          | 0/534 [00:00<?, ?it/s]

Epoch 9 -> Train Acc: 99.45% | Val Acc: 98.81%


Epoch 10 Train:   0%|          | 0/534 [00:00<?, ?it/s]

Epoch 10 -> Train Acc: 99.53% | Val Acc: 99.06%

Model saved perfectly to: /content/drive/MyDrive/Malware_Project/binary_malware_scanner.pth
